
Tasks:
- Build a second index using SentenceWindowNodeParser — 3 sentence window
- Run your same 5 queries on both indexes
- For each: which chunking returned more relevant context and why?
- Write 3–4 bullet points on when fixed chunking fails for complaint records


Deliverable: "chunking comparison" markdown section with side-by-side outputs and written observations.




In [ ]:
import csv # built - in CSV reader -> parse complaints.csv row by row
from pathlib import Path # modern cross-platform path handling
from dotenv import load_dotenv #reads a .env file and dumps keys into os.environ

#RAG stack=> Llamaindex + FAISS 

import faiss# facebook's vector db + stores 384 dim vectors + find nearest neighbor fast

from llama_index.core import Document, Settings, VectorStoreIndex, StorageContext
#Document -> wraps ONE row(text+metadata) so Llamaindex can consume it => labeled tupperware holding ingredient(text) + its label(metadata)
#Settings -> global config bag(set embedder/LLM once, whole framework picks it up) => rulebook only use olive oil
#VectorStoreIndex-> the orchestrator(takes docs-> embeds-> stores in the vector DB)=> what procedure to follow 
#StorageContext -> a routing config: "puts vectors into this specific FAISS instance"

from llama_index.vector_stores.faiss import FaissVectorStore
#ADapter class => lets LlamaIndex (python high level) talk to FAISS (low level c++) without it they cant communicate - different shap[es they have come to common grounds]

from llama_index.llms.groq import Groq
#llamaindex's adapter for Groq's LLM API

##Chunking and Preprocessing

from llama_index.core.node_parser import SentenceWindowNodeParser
#chunker we test against the default splits text into INDIVIDUAL SENTENCES(one node per sentence), but attaches surrounding sentences as a "window" in metadata
from llama_index.core.postprocessor import MetadataReplacementPostProcessor
#Runs AFTER retrieval b4 sending to LLM swaps each retrieved node's text for its "window" metadata, so LLM sees the boroader context instead of just one isolated sentence(3 + 3)

##loqad environment variables
load_dotenv()
#reads .env file -> sets os.environ["Groq_API_KEY"] 



True

In [ ]:
##Global setting = set once
Settings.embed_model=HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
#both indexes (fixed + window) will use the SAME embedder. If we used different embedders we couldn't tell whether quality  differences came from chunking or from embedder-> confounding variable

Settings.llm=Groq(
    model="llama-3.3-70b-versatile",
    temperature=0# deterministic output we dont want creative response
)
#same llm for both query engines 


### Load Documents

In [8]:
def load_documents():#everyhting wrapped in a function for reusability + clarity
    """
    Read complaints.csv row by row-> return a list of LlamaIndex Documents.

    Each Document wraps:
        text = the complaint text (this is what gets embedded)
        metadata = extra fild we want back with each retrieved result(NOT embedded stored as is alongside vector)
    """
    CSV_PATH=Path("complaints.csv")
    #Path object not string 

    docs=[]
    with open(CSV_PATH, "r") as f:
        #using with auto close if an error comes wo it u owuld require f.close and all

        for i, row in enumerate(csv.DictReader(f)):
            #csv.Dictreader reads first line as column headers -> every subsequent row becomes a dict: row["complaints_text"] instead of row[0] enumerate gives us (index, row) pairs. WE use i as a stable row_id in metadata

            docs.append(Document(
                text=row["complaint_text"],
                #only field that gets embedded into a vector 

                metadata={
                    "product_area": row["product_area"],
                    "root_cause_label": row["root_cause_label"],
                    "capa_summary": row["capa_summary"],
                    "row_id":i
                }
            )) 
    return docs

#call it once save to modeule-level variable so both indexes can reuse it
documents=load_documents()
print(f"Loaded {len(documents)} documents.")

Loaded 71 documents.


In [9]:
print(documents[0])
print(documents[0].text[:100])
print(documents[0].metadata)


Doc ID: 3d9cb7ad-ffd0-400f-a318-68a0f8639e8c
Text: Incoming batch of steel fasteners failed tensile strength
testing — 23% below spec tolerance. Supplier lot #BT-4421.
Incoming batch of steel fasteners failed tensile strength testing — 23% below spec tolerance. Suppli
{'product_area': 'Fasteners', 'root_cause_label': 'supplier_defect', 'capa_summary': 'Issued SCAR to supplier. Quarantined remaining lot. Added incoming tensile test to sampling plan.', 'row_id': 0}


### Build 2 indexes
- **index_fixed**: Llamaindex default chunker for short complaints produce roughly 1 node per row
- **index_window**: SentenceWindowParser. One node per sentence. Each node carries a 3 -sentence window as metadata for later retrieval

**Controlled Variable** (same for both): input docs, embedding model, LLm, FAISS metric(L2)
**Independent Variable**(the one thing that differs): chunking startegy

In [ ]:
def build_fixed_index(docs):
    """
    BASELINE index using LlamaIndex's default chunker.

    Defauklt chunker = SentenceSplitter (fixed-size, `1024 tokens per chunk),
    for short complaints (~50 tokens each) whole complaoint fits in 1 chunk -> expect ~71 nodes for 71 documents
    """

    faiss_index=faiss.IndexFlatL2(384)
    # Index => FAISS's word for vector DB
    # Flat => brute force, stores vector as is no compression fine for 71 vector
    #L2=> euclidean distance
    #384=> vector dimension in case of MIniLM's output size exactly

    #wrap the raw FAISS in LlamaIndex's adapter (faiss low level c++ can talk to LLamaIndex (python) using this)
    vector_store=FaissVectorStore(faiss_index=faiss_index)

    #tell LlamaIndex: "when you need to store vectors put em in this vector instance store other options=> doc_stor , imdex_store"
    storage_context=StorageContext.from_defaults(vector_store=vector_store)

    #embed every doc and insert into FAISS
    #default sentence splitter for LlamaIndex
    index=VectorStoreIndex.from_documents(
        docs,
        storage_context=storage_context,
        show_progress=True
    )

    print(f"[FIXED] Total vectors stored: {faiss_index.ntotal}")
    # .ntotal =number of vectors FAISS is holding for 71 short docs->` 71
    return index


index_fixed=build_fixed_index(documents)

Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/71 [00:00<?, ?it/s]

[FIXED] Total vectors stored: 71


In [ ]:
def build_window_index(docs, window_size=3):
    """
    EXPERIMENTAL index using SentenceWindowNodeParser,

    each doc is split into sentences ecah sentence becomes its own node . Each mode carries a 'window' of surrounding sentences as metadata {window_size=3 means 3 sentence b4 and 3 sentences after}

    Expected node count: MORE than 71 bcoz multi-sentence complaints produce multiple nodes
    """

    #parser is the key differnce swaps default chunker
    parser=SentenceWindowNodeParser.from_defaults(
        window_size=window_size,
        #window_size=3 => each node sees itself + 3 before + 3 after = upto 7 sentences task specifies 3

        window_metadata_key="window",
        # the dict key under which window gets stored on each node. When we query postprocessor look for this eaxct key
        #         node.text = "S3"                           # single sentence (gets embedded)
        # node.metadata["window"] = "S1 S2 S3 S4 S5" # the surrounding context

        original_text_metadata_key="original_text",
        #stores the node's og single sentence seperately useful for debugging - lets see what matched vs what got expkanded
        # node.text = "S3"                              # gets embedded
        # node.metadata["window"] = "S1 S2 S3 S4 S5"   # expanded context
        # node.metadata["original_text"] = "S3"         # backup copy
    )

    #Fresh faiss - seperate form above one as faiss_index from above may collide=> faiss is vector store holds vector
    faiss_index=faiss.IndexFlatL2(384)
    vector_store=FaissVectorStore(faiss_index=faiss_index)
    storage_context=StorageContext.from_defaults(vector_store=vector_store)

    #key diff from build_fixed_index: transformations=[parser] => this tells VectorStoreIndex: "b4 embedding, run docs thru this parser"
    index=VectorStoreIndex.from_documents(
        docs,
        storage_context=storage_context,
        transformations=[parser], #here 2 indexes will differ
        show_progress=True
    )
    #for each doc in docs: 
        # 1. Apply tranformations(chunkers)-> get a list of nodes
        # 2. for each node: 
            # a. call embedder-> get 384-dim vector 
            # b. Send vector + metadata to the vector_store(FAISS) 
            # c. Also store node in doc_store 
        # 3. Update index_store so you can retrieve by node id

    print(f"[WINDOW] Toltal vectors stored: {faiss_index.ntotal}")
    #expect larger number than fixed ijndex
    #faiss.ntotal=> would copntain BOTH sets of vectors mixed together if we dont use fresh faiss_index
    return index   # ← hand the built index back to the caller 

index_window=build_window_index(documents, window_size=3)

Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/109 [00:00<?, ?it/s]

[WINDOW] Toltal vectors stored: 109


### Step 3: Wrap each index in a query engine
- **'qe_fixed'**-plain query engine, 'similarity_top_k=3
- **'qe_window'** - same 'top_k=3' **plus** 'MetadataReplacementPostProcessor'
The post processor swaps each retrieved node's text for its 'window' metadata b4 sending to LLm. Without it, LLm sees only single sentences

then we define 'ask_side_by_side'- runs the same question on both engines and prints answers+ sources

In [ ]:
#QUERY ENGINE #1 - FIXED CHUNKING
qe_fixed = index_fixed.as_query_engine(
    similarity_top_k=3,
    #when answering a question retrieve 3 most similar cunks 
    #sam evalue for the both engines=> fair compariosn
    #method=> glues 4 things retriever derived from the index + LLM (from settings.llm) + prompt template with placeholder for context and question + a response synthesizer that formats final output 
)

#QUERY ENGINE #2 - SentenceWindow (needs the postprocessor!)
qe_window = index_window.as_query_engine(
    similarity_top_k=3,
    #postprocessors are pipeline run in order takes list of retrieved nodes trnasform them and passes nto the next
    node_postprocessors=[
        MetadataReplacementPostProcessor(target_metadata_key="window")
        #target_metadata_key="window" mUST match the window_metadata_key="window" we set when creating SentenceWindowNodeParser if these not match=> postprocessor does nothing They're referring to the same slot on the node:

# Block 3 (parser): "store the window under the key 'window'"
# Block 4 (postprocessor): "look for the window under the key 'window'"

    ]
    # Flow with postprocessor in place 
        # 1. Question-> embed 
        # 2. FAISS returns top 3 sentence-nodes(each node.text=ONE sentence) 
        # 3. Postprocessor runs b4 LLM: 
            #   for each retrieved node: 
            #       node.text=node.metadata["window"] 
        # 4. LLM now sees the multi-sentence windows -> synthesize answer 
    # Withotu the postprocessor: Step 3 is skipped LLM sees only single sentences

)

#side by side runner 
def ask_side_by_side(question):
    """
    Run one question through Both engines. Print answer + sources from each. why to print it? beacuse we want to check whether the answer was grounded against what was given to llm. 
    """
    print("="*80)
    print(f"QUESTION: {question}")
    print("=" *80)

    for label, engine in [("FIXED CHUNKING", qe_fixed),
                          ("SENTENCE WINDOW (window_size=3)", qe_window)]:
        print(f"\n----------{label}----------")

        response=engine.query(question)
        #engine.query(..) runs the whole pipeline in one call:
             #  embed question-> retrieve top_k-> (postprocess if any)-> LLM-> Response 
        # Returns a response object with: 
        #   str(response)-> the generated answer text 
        #   reponse.source_nodes-> the retrieved chunks what LLM saw

        print(f"\nSOURCES ({len(response.source_nodes)} retrieved):")
        for i, node in enumerate(response.source_nodes,1):
            print(f"\n [{i}] score={node.score:.3f} "
                  f"root_cause={node.metadata.get('root_cause_label','N/A')}")
            #Fow the window engine, node.txt has been swapped by postprocessor to be the window (multi-sentence context). So printing node.text here shows whatb the LLM actually saw- good for grounding analysis
            print(f"  {node.text[:250]}...")

    print("\n")

In [17]:
print("qe_fixed  postprocessors:", qe_fixed._node_postprocessors if hasattr(qe_fixed, '_node_postprocessors') else 'N/A (OK for default)')
print("qe_window postprocessors:", [type(p).__name__ for p in qe_window._node_postprocessors])


qe_fixed  postprocessors: []
qe_window postprocessors: ['MetadataReplacementPostProcessor']


### Run the 5 queries + observation

In [18]:
ask_side_by_side("Have we seen supplier issues with PCB boards before?")


QUESTION: Have we seen supplier issues with PCB boards before?

----------FIXED CHUNKING----------

SOURCES (3 retrieved):

 [1] score=0.972 root_cause=supplier_defect
  PCB boards from Supplier B have incorrect trace routing — does not match approved Gerber files revision 3.2....

 [2] score=1.042 root_cause=ambiguous
  Customer reports burnt smell from power supply unit. Inspection shows discoloration on PCB near voltage regulator. Could be component rated too low (design), counterfeit component from supplier, or solder bridge from assembly process....

 [3] score=1.132 root_cause=human_error
  Solder paste stencil for Product X used on Product Y assembly. Wrong pad pattern — 100 boards affected....

----------SENTENCE WINDOW (window_size=3)----------

SOURCES (3 retrieved):

 [1] score=0.972 root_cause=supplier_defect
  PCB boards from Supplier B have incorrect trace routing — does not match approved Gerber files revision 3.2....

 [2] score=1.097 root_cause=ambiguous
  Customer rep

Tie - both retrieved the samne top supplier defect complaint whole complaint embedding and sentence-level retrieval coverged on the same best match because the relevant complaint is single sentence

In [19]:
ask_side_by_side("What equipment failures have we had with hydraulic presses?")


QUESTION: What equipment failures have we had with hydraulic presses?

----------FIXED CHUNKING----------

SOURCES (3 retrieved):

 [1] score=0.807 root_cause=equipment_failure
  Hydraulic press #3 failed mid-cycle — lost pressure at 80% of required forming force. 15 parts deformed....

 [2] score=1.128 root_cause=equipment_failure
  Air compressor delivering 85 PSI instead of required 100 PSI. Pneumatic actuators underperforming on press line....

 [3] score=1.174 root_cause=equipment_failure
  CNC spindle bearing seized during production run. Tool crashed into workpiece. 3 parts destroyed....

----------SENTENCE WINDOW (window_size=3)----------

SOURCES (3 retrieved):

 [1] score=0.852 root_cause=equipment_failure
  Hydraulic press #3 failed mid-cycle — lost pressure at 80% of required forming force.  15 parts deformed....

 [2] score=0.964 root_cause=equipment_failure
  Hydraulic press #3 failed mid-cycle — lost pressure at 80% of required forming force.  15 parts deformed....

 [3]

window won #3 more closer boz sentence level embedding caught the specific phrase 'press line'

In [20]:
ask_side_by_side("Have there been any design issues causing plastic parts to crack?")


QUESTION: Have there been any design issues causing plastic parts to crack?

----------FIXED CHUNKING----------

SOURCES (3 retrieved):

 [1] score=1.159 root_cause=design_issue
  Weld joint on bracket cracked during vibration testing but passed static load test — fatigue life not considered in design....

 [2] score=1.162 root_cause=supplier_defect
  Rubber O-rings received from vendor showed visible surface cracks before installation. Entire shipment rejected....

 [3] score=1.175 root_cause=ambiguous
  Plastic enclosure warping after 6 months in field — could be residual stress from molding process (supplier), insufficient wall thickness (design), or environmental exposure beyond rated conditions (customer misuse)....

----------SENTENCE WINDOW (window_size=3)----------

SOURCES (3 retrieved):

 [1] score=1.151 root_cause=supplier_defect
  Rubber O-rings received from vendor showed visible surface cracks before installation.  Entire shipment rejected....

 [2] score=1.159 root_cause

Fixed wins clearly. Fixed's [1] was "Weld joint cracked... fatigue life not considered in design" — a design_issue about cracking, perfect. Window's [1] was "O-rings showed surface cracks" — cracking yes, but O-rings aren't plastic, and it's supplier_defect not design. The single sentence "O-rings showed cracks" matched on the word "cracks" without the broader design context that whole-complaint embedding preserved.

"Fixed won — top match was an actual design_issue about cracking (weld joint fatigue). Window's top match drifted to a supplier O-ring complaint that matched on the word 'cracks' alone, missing the 'design' angle."



In [21]:
ask_side_by_side("What human errors have occurred on CNC machines?")


QUESTION: What human errors have occurred on CNC machines?

----------FIXED CHUNKING----------

SOURCES (3 retrieved):

 [1] score=0.968 root_cause=human_error
  Operator loaded incorrect program on CNC machine — Part #A vs Part #B. 12 parts machined wrong before detection....

 [2] score=0.979 root_cause=human_error
  Technician replaced sensor with wrong part number — similar form factor but different measurement range....

 [3] score=1.030 root_cause=equipment_failure
  CNC spindle bearing seized during production run. Tool crashed into workpiece. 3 parts destroyed....

----------SENTENCE WINDOW (window_size=3)----------

SOURCES (3 retrieved):

 [1] score=0.979 root_cause=human_error
  Technician replaced sensor with wrong part number — similar form factor but different measurement range....

 [2] score=0.993 root_cause=human_error
  Operator loaded incorrect program on CNC machine — Part #A vs Part #B.  12 parts machined wrong before detection....

 [3] score=1.053 root_cause=equi

Fixed wins clearly. Fixed's [1] was "Operator loaded incorrect program on CNC machine" — textbook perfect match to the query. Window's [1] was "Technician replaced sensor with wrong part number" — human_error, but not CNC-specific. Sentence-level matching missed that the CNC context is distributed across the full complaint, not concentrated in any single sentence.

"Fixed won — retrieved the CNC-program-loading complaint as top match. Window picked a human_error complaint unrelated to CNC because no individual sentence in the CNC complaint embedded as strongly as its whole."



In [22]:
ask_side_by_side("Are there complaints where the root cause was unclear or spanned multiple categories?")


QUESTION: Are there complaints where the root cause was unclear or spanned multiple categories?

----------FIXED CHUNKING----------

SOURCES (3 retrieved):

 [1] score=1.417 root_cause=design_issue
  PCB trace width insufficient for actual current load — trace heating observed during extended operation testing....

 [2] score=1.420 root_cause=design_issue
  Wire routing in harness assembly creates sharp bend radius at connector exit — field failures from conductor fatigue....

 [3] score=1.426 root_cause=supplier_defect
  Vendor-supplied labels have incorrect UL certification marking — wrong file number printed on 10,000 labels....

----------SENTENCE WINDOW (window_size=3)----------

SOURCES (3 retrieved):

 [1] score=1.417 root_cause=design_issue
  PCB trace width insufficient for actual current load — trace heating observed during extended operation testing....

 [2] score=1.420 root_cause=design_issue
  Wire routing in harness assembly creates sharp bend radius at connector exit — 

Ambiguous / cross-category
Both failed equally. Scores are all >1.4 (high = far = poor matches). Neither engine retrieved any complaint actually labeled root_cause=ambiguous. The word "ambiguous" is a category label, not vocabulary that appears in the complaint text itself. Both chunking strategies fail — this is a retrieval problem, not a chunking problem. You'd need metadata filtering (root_cause_label == "ambiguous") instead of semantic search.

"Both failed — neither retrieved any ambiguous-labeled complaints. The failure is semantic: 'ambiguous' is a metadata label, not text that appears in complaints, so no chunking strategy can rescue it. Metadata filtering would."



Complaints are already short → whole-complaint embedding captures the gist as well or better than sentence-level retrieval in 4/5 cases
Window only wins when a critical phrase (like "press line") is buried inside a larger complaint that averages out
Window can lose when a concept is distributed across a whole complaint (like "CNC" + "operator error") and no single sentence carries the full signal
Neither helps for metadata-style questions — Q5 shows retrieval-over-text fundamentally can't answer "which complaints have category X"
Fixed chunking is the safer default for this dataset. You'd only reach for window chunking if your complaints were much longer (multi-paragraph CAPA reports, audit findings, etc.)


## When fixed chunking fails for complaint records — observations

Based on running 5 queries through both `index_fixed` (default SentenceSplitter) and 
`index_window` (SentenceWindowNodeParser, window_size=3) over 71 manufacturing complaints:

- **Fixed chunking is the right default for this dataset.** Complaints average 1–3 sentences each; the default SentenceSplitter packs an entire complaint into one chunk, so whole-complaint embedding captures the full semantic signal. Fixed won or tied 4 out of 5 queries (Q1, Q3, Q4; tied Q5).

- **Fixed chunking loses when a specific phrase is buried inside a multi-sentence complaint.** In Q2 (hydraulic presses), window chunking surfaced a complaint containing "press line" in a later sentence — whole-complaint embedding averaged this phrase away, while sentence-level embedding matched it directly. Fixed fails when the query's keyword is a minority signal in a longer complaint.

- **Fixed chunking is fine when the concept is distributed across the whole complaint.** In Q4 (CNC human errors), the relevant complaint "Operator loaded incorrect program on CNC machine" carries its meaning across the whole text; no single sentence dominates. Window chunking fragmented the signal and retrieved a less relevant complaint. Fixed *succeeds* here precisely because it doesn't split.

- **Neither chunking strategy helps when the query targets metadata, not text.** In Q5 (complaints with ambiguous root causes), both engines failed — no complaint in the dataset uses the word "ambiguous" in its text; it only exists as a metadata label. This is a retrieval-layer problem that requires metadata filtering, not a chunking fix. If real QMS queries often target categories ("show all supplier_defect issues from 2024"), chunking experiments won't solve them — a metadata-aware retriever will.

### Overall verdict

For this dataset (71 short complaints, 1–3 sentences each): **ship fixed chunking.** 
SentenceWindowNodeParser would become valuable if the records grew into long-form docs — 
full CAPA reports, audit findings, investigation narratives, or multi-page SOPs — where a 
query's answer lives in one paragraph of a 10-page document. On short complaint records, 
the overhead of sentence-level indexing and the metadata replacement postprocessor delivers 
no meaningful gain and occasionally hurts retrieval quality.


Wire the retriever to an LLM response. Observe, don't just run.


Tasks:
- Connect the LlamaIndex query engine to GPT-4o-mini or Claude
- Run 5 queries covering different root cause categories
- For each: was the answer grounded or hallucinated? Write one sentence per query
- Try similarity_top_k = 3 vs 6 — does quality improve or get noisier?


Deliverable: Notebook with 5 query results + a markdown cell per query with your 1-sentence observation.


### Top_K comparison 

using **index_fixed** we now vary only the 'similarity_top_k' knob to see whether retrieving MORE context helps the LLM write better answers or just adds noise

**The experimental quetsion:**
- k=3 = LLM sees the 3 most similar complaints per question
- k=6 = LLM sees the 6 most similar complaints per question
- More context -> more complete answers, OR more distraction -> fuzzier answers?

**The grading rubric for each query: "Grounded or Hallucination"**
- *Grounded* - every specific claim in the answer traceable to atleast one retrieved source
- *Hallucinated* - answer invents specific

In [25]:
#=======================================================
# TWO QUERY ENGINES - identical except for top_k
# Both use index_fixed (Part 1 verdict: fixed chunking wins for this dataset) 
# Both use Settings.llm 
# Only the retrieval breadth differs

#=======================================================

qe_k3 = index_fixed.as_query_engine(similarity_top_k=3)
#retrieves the 3 most similar complaints per question Narrow=> LLm sees a tight focused context Risk: might miss a 4th complaint was relevant

qe_k6 = index_fixed.as_query_engine(similarity_top_k=6)
#wider-LLM has more wiork to do - introduce noise too


#side by side comparison
def ask_topk_compare(question):
    """
    Run the SAME question through both engines.
    print answer + sources for each so we can judge:
        - Is the answer frounded? (every cliam traceable to a source?)
        - Does k=6 add value or noise?
    """

    print("="*80)
    print(f"QUESTION: {question}")
    print("=" * 80)

    for label, engine in [("top_k=3", qe_k3), ("top_k=6", qe_k6)]:
        print(f"\n------ {label} -------")
        response=engine.query(question)
        #Pipeline: embed question-> FAISS top-k -> stuff into prom pt -> LLM -> Response

        # The LLM's answer
        print(f"\nANSWER:\n{response}")

        #The sources the LLM was given - REQUIRED for grounding judgement
        print(f"\nSOURCES ({len(response.source_nodes)} retreived):")
        for i, node in enumerate(response.source_nodes,1):
            print(f" [{i}] score={node.score: .3f}"
                f"| root_cause={node.metadata.get('root_cause_label', 'N/A')}"
                f"| product={node.metadata.get('product_area', 'N/A')}"
                )
            #Truncate to 180 chars - k=6 prints 6 sources so we wnat compact op
            print(f"  {node.text[:180]}...")
    print()


            
                


### Queries

In [26]:
# ─── Query 1 — supplier_defect category ───
# Target: a complaint about supplier-caused defects on PCB boards.
# We picked this category because Part 1 showed strong retrieval on PCB/supplier
# queries (Q1 in chunking comparison) — a good "easy case" to start with.
ask_topk_compare("Have we seen supplier issues with PCB boards before?")


QUESTION: Have we seen supplier issues with PCB boards before?

------ top_k=3 -------

ANSWER:
Yes, there was an issue with PCB boards from Supplier B having incorrect trace routing that did not match approved Gerber files. This issue was addressed by quarantining the affected boards, issuing a deviation report, and re-qualifying the supplier with an updated verification step.

SOURCES (3 retreived):
 [1] score= 0.972| root_cause=supplier_defect| product=Electronics
  PCB boards from Supplier B have incorrect trace routing — does not match approved Gerber files revision 3.2....
 [2] score= 1.042| root_cause=ambiguous| product=Electronics
  Customer reports burnt smell from power supply unit. Inspection shows discoloration on PCB near voltage regulator. Could be component rated too low (design), counterfeit component ...
 [3] score= 1.132| root_cause=human_error| product=Electronics Assembly
  Solder paste stencil for Product X used on Product Y assembly. Wrong pad pattern — 100 boards

### Q1 — supplier_defect: "Have we seen supplier issues with PCB boards before?"

**Grounded or hallucinated?**  
Both answers are grounded, but k=3 produced a meaningfully *better* answer — 
it named Supplier B, the Gerber file issue, and the specific CAPA (quarantine + 
deviation report + re-qualification). k=6 retrieved 3 tangentially-related PCB 
complaints that caused the LLM to generalize to "suppliers" plural, drop the 
specific CAPA, and surface a weakly-related "counterfeit components" mention 
from source [2]. k=3 wins on specificity; k=6 added noise that diluted the answer.


In [27]:
# ─── Query 2 — equipment_failure category ───
# Target: failures of a specific piece of equipment (hydraulic press).
# Why this query: tests whether the LLM stays focused on ONE equipment type
# when the dataset contains many equipment-related complaints (compressors,
# CNC bearings, conveyor motors). k=6 is at risk of pulling in non-press failures.
ask_topk_compare("What equipment failures have we had with hydraulic presses?")


QUESTION: What equipment failures have we had with hydraulic presses?

------ top_k=3 -------

ANSWER:
One hydraulic press, specifically press #3, failed mid-cycle due to a lost pressure issue at 80% of the required forming force, which was caused by a failed hydraulic pump seal.

SOURCES (3 retreived):
 [1] score= 0.807| root_cause=equipment_failure| product=Stamping
  Hydraulic press #3 failed mid-cycle — lost pressure at 80% of required forming force. 15 parts deformed....
 [2] score= 1.128| root_cause=equipment_failure| product=Pneumatics
  Air compressor delivering 85 PSI instead of required 100 PSI. Pneumatic actuators underperforming on press line....
 [3] score= 1.174| root_cause=equipment_failure| product=Machining
  CNC spindle bearing seized during production run. Tool crashed into workpiece. 3 parts destroyed....

------ top_k=6 -------

ANSWER:
One equipment failure occurred with a hydraulic press, specifically Hydraulic press #3, which failed mid-cycle due to a lost press

### Q2 — equipment_failure: "What equipment failures have we had with hydraulic presses?"

**Grounded or hallucinated?**  
Both grounded, but **k=6 won on completeness** — its answer included the CAPA 
details (pump seal replaced, predictive maintenance schedule added) that k=3 
left out. The 3 extra sources at k=6 (weld joint, conveyor, PLC) were all 
non-press-related and were correctly ignored by the LLM; they didn't dilute 
the answer because source [1] dominated (score 0.807 vs next 1.128). 
Opposite pattern from Q1 — suggests k=6 is safer when the top source is 
significantly stronger than the rest.


In [29]:
# ─── Query 3 — design_issue category ───
# Target: design-caused cracking in plastic parts.
# Why this query: deliberately probes a DIFFICULT retrieval case. The CSV has
# cracking complaints across multiple root causes (supplier O-ring cracks,
# design fatigue failures, weld cracks). The query specifies BOTH the root cause
# ("design issues") AND the material ("plastic") AND the symptom ("crack").
# Tests whether retrieval filters on all three — or just matches on "crack".
ask_topk_compare("Have there been any design issues causing plastic parts to crack?")


QUESTION: Have there been any design issues causing plastic parts to crack?

------ top_k=3 -------

ANSWER:
Yes, there was an issue with a weld joint on a bracket cracking during vibration testing due to a design issue, as fatigue life was not considered in the design. Additionally, there was an issue with a plastic enclosure warping, which could be due to insufficient wall thickness, a potential design issue, although the root cause is listed as ambiguous.

SOURCES (3 retreived):
 [1] score= 1.159| root_cause=design_issue| product=Structural Components
  Weld joint on bracket cracked during vibration testing but passed static load test — fatigue life not considered in design....
 [2] score= 1.162| root_cause=supplier_defect| product=Sealing Components
  Rubber O-rings received from vendor showed visible surface cracks before installation. Entire shipment rejected....
 [3] score= 1.175| root_cause=ambiguous| product=Enclosures
  Plastic enclosure warping after 6 months in field — coul

### Q3 — design_issue: "Have there been any design issues causing plastic parts to crack?"

**Grounded or hallucinated?**  
**k=3 hallucinated, k=6 nailed it.** k=3 produced a Frankenstein answer that 
combined the weld-crack (design + metal) with plastic-enclosure-warping 
(plastic + ambiguous + not cracking) to confidently say "yes" — no single 
source actually matched all three constraints (plastic + design + crack). 
k=6 correctly said "no" and walked through each near-miss complaint (O-rings = 
supplier, resin = supplier, weld = not plastic, enclosure = warping not cracking), 
using the broader context to rule out rather than rationalize. A rare case 
where k=6 clearly wins on answer correctness, not just completeness.


In [30]:
# ─── Query 4 — human_error category ───
# Target: operator mistakes on CNC machines.
# Why this query: Part 1's chunking comparison showed this was a query where
# WHOLE-complaint embedding beat sentence-level embedding (the CNC concept is
# distributed across the full complaint, not concentrated in any one sentence).
# Here we hold chunking constant (fixed) and vary only top_k — does k=6 help
# or hurt when the ideal match is one specific complaint?
ask_topk_compare("What human errors have occurred on CNC machines?")


QUESTION: What human errors have occurred on CNC machines?

------ top_k=3 -------

ANSWER:
Two human errors have occurred on CNC machines. The first error was when an operator loaded the incorrect program on the CNC machine, resulting in 12 parts being machined wrong. The second error was not directly related to the CNC machine itself, but rather a technician replacing a sensor with the wrong part number, which could potentially affect the CNC machine's operation. However, the first error is the one that directly involved the CNC machine.

SOURCES (3 retreived):
 [1] score= 0.968| root_cause=human_error| product=Machining
  Operator loaded incorrect program on CNC machine — Part #A vs Part #B. 12 parts machined wrong before detection....
 [2] score= 0.979| root_cause=human_error| product=Maintenance
  Technician replaced sensor with wrong part number — similar form factor but different measurement range....
 [3] score= 1.030| root_cause=equipment_failure| product=Machining
  CNC spind

### Q4 — human_error: "What human errors have occurred on CNC machines?"

**Grounded or hallucinated?**  
Both grounded, **k=6 won on completeness** — it retrieved and cited the 
drawing-revision error (Rev C vs Rev E, 25 parts machined) that sits at rank 5 
and was invisible to k=3. The LLM at k=6 correctly filtered out 3 non-CNC 
human errors (sensor replacement, solder stencil, vision camera) rather than 
forcing them into the answer. k=3 was forced to hedge on the sensor 
replacement because it had no better second example in its top 3.


In [31]:
# ─── Query 5 — ambiguous / cross-category ───
# Target: complaints that themselves span multiple root causes (the CSV has
# records 67-72 labeled root_cause_label="ambiguous" — complaints where the
# text explicitly says "could be supplier OR design OR process").
# Why this query: it probes a FUNDAMENTAL RAG weakness — the question targets
# a METADATA LABEL, not vocabulary that appears in complaint text. 
# Part 1 showed both engines failed here. Does k=6 rescue it or confirm failure?
ask_topk_compare("Are there complaints where the root cause was unclear or spanned multiple categories?")


QUESTION: Are there complaints where the root cause was unclear or spanned multiple categories?

------ top_k=3 -------

ANSWER:
No, the root cause for each complaint is specified and falls under a single category, such as design_issue or supplier_defect. There are no instances where the root cause is unclear or spans multiple categories.

SOURCES (3 retreived):
 [1] score= 1.417| root_cause=design_issue| product=Electronics
  PCB trace width insufficient for actual current load — trace heating observed during extended operation testing....
 [2] score= 1.420| root_cause=design_issue| product=Wiring
  Wire routing in harness assembly creates sharp bend radius at connector exit — field failures from conductor fatigue....
 [3] score= 1.426| root_cause=supplier_defect| product=Labeling
  Vendor-supplied labels have incorrect UL certification marking — wrong file number printed on 10,000 labels....

------ top_k=6 -------

ANSWER:
No, the root cause for each complaint is clearly identified 

### Q5 — ambiguous: "Are there complaints where the root cause was unclear or spanned multiple categories?"

**Grounded or hallucinated?**  
Both answers were **grounded in their retrieved sources but globally wrong** — 
they confidently said "no ambiguous complaints exist" despite the dataset 
containing several (the word "ambiguous" is in metadata, not complaint text, 
so semantic retrieval missed them; all scores >1.4 confirm weak matches). 
k=6 returned the same failure as k=3 — this is a retrieval-layer problem 
that no top_k tweak can fix. The real solution is metadata filtering 
(`root_cause_label == "ambiguous"`) alongside semantic search — a hybrid retriever.


Query	Winner	Category	Why
Q1 — PCB suppliers	k=3	supplier_defect	k=6 diluted specific CAPA into generic plural
Q2 — hydraulic presses	k=6	equipment_failure	Longer context surfaced more CAPA detail from top source
Q3 — plastic cracking	k=6	design_issue	Extra context let LLM correctly say "no match" instead of Frankenstein
Q4 — CNC human error	k=6	human_error	Relevant 2nd example at rank 5 surfaced at k=6
Q5 — ambiguous	Tie (both failed)	ambiguous	Retrieval-layer failure, not fixable by top_k